# 00 — Filter the full West Virginia FIA download (run once, locally)
Reads every `WV_*.csv` in the FIA folder **in chunks** (never loads a whole file into memory), keeps only the plots,
inventory years and columns the Fernow analyses use, and writes compressed copies (`.csv.gz`, read natively by pandas)
plus two small condition-level summary tables. Run in ArcGIS Pro Notebooks; default `arcgispro-py3` environment.

**What it keeps**
- Plots in scope (`SCOPE`, default all of WV) measured in or after `MIN_INVYR` (default 2000, so annual-inventory
  remeasurements can still link to their previous visit).
- All columns of small tables; a column subset of the big tree tables (`TREE`, `TREE_GRM_*`).
- Rows linked to kept plots via `PLT_CN` (or to kept trees via `TRE_CN`). Tables with neither key are copied whole if
  small, otherwise skipped and reported.

**What it builds**
- `fia_cond_live.csv.gz` — per plot × condition: live aboveground/belowground tree C, live basal area, stems, C in
  large trees (≥50 cm DBH), sapling density, standing-dead C — per acre **of condition** and in Mg ha⁻¹.
- `fia_cond_seedlings.csv.gz` — per plot × condition: seedlings per acre by species group (oak, red maple, sugar maple,
  beech, birch, black cherry, yellow-poplar, striped maple, hickory, other).
- `MANIFEST.csv` — rows and MB in/out for every file, so the reduction is documented.


In [ ]:
# ============================== CONFIG ==============================
import os, glob, time
IN_DIR  = r'C:\Users\DCifelli2\Downloads\WV_CSV'            # full FIA download
OUT_DIR = r'C:\Users\DCifelli2\Downloads\WV_CSV_filtered'   # compressed, filtered output (copy into the repo)

MIN_INVYR = 2000                 # keep 2000 so 2004+ annual plots can link to their previous visit
SCOPE = 'WV'                     # 'WV' (default, all plots) | 'SECTION' (ECOSUBCD section, e.g. M221B) | 'RADIUS'
SECTION = 'M221B'                # used when SCOPE == 'SECTION'
FERNOW_LATLON, RADIUS_KM = (39.06, -79.68), 60   # used when SCOPE == 'RADIUS'

SKIP = ['TREE_GRM_ESTN', 'OZONE']      # substrings of tables not needed (GRM_ESTN is enormous)
COPY_WHOLE_MAX_MB = 50           # tables with no plot/tree key are copied whole only if smaller than this
CHUNK = 500_000

# Column subsets for the big tables (others keep every column). Missing columns are ignored, so FIADB version changes are safe.
COLS = {
    'TREE': ['CN', 'PLT_CN', 'PREV_TRE_CN', 'INVYR', 'CONDID', 'SUBP', 'TREE', 'STATUSCD', 'SPCD', 'SPGRPCD', 'DIA', 'HT',
             'ACTUALHT', 'CR', 'CCLCD', 'TREECLCD', 'AGENTCD', 'STANDING_DEAD_CD', 'DECAYCD', 'MORTYR', 'PREVDIA',
             'RECONCILECD', 'TPA_UNADJ', 'DRYBIO_AG', 'DRYBIO_BG', 'CARBON_AG', 'CARBON_BG'],
    'TREE_GRM_COMPONENT': ['TRE_CN', 'PREV_TRE_CN', 'PLT_CN', 'STATECD', 'DIA_BEGIN', 'DIA_MIDPT', 'DIA_END',
                           'SUBP_COMPONENT_AL_FOREST', 'SUBP_SUBPTYP_GRM_AL_FOREST', 'SUBP_TPAGROW_UNADJ_AL_FOREST',
                           'SUBP_TPAREMV_UNADJ_AL_FOREST', 'SUBP_TPAMORT_UNADJ_AL_FOREST'],
    'TREE_GRM_BEGIN': ['TRE_CN', 'PLT_CN', 'DIA', 'CARBON_AG', 'CARBON_BG', 'DRYBIO_AG', 'DRYBIO_BG'],
    'TREE_GRM_MIDPT': ['TRE_CN', 'PLT_CN', 'DIA', 'STATUSCD', 'CARBON_AG', 'CARBON_BG', 'DRYBIO_AG', 'DRYBIO_BG'],
}
# ====================================================================
import numpy as np, pandas as pd
LBAC_TO_MGHA = 0.00112085
os.makedirs(OUT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(IN_DIR, '*.csv')))
inv = pd.DataFrame({'file': [os.path.basename(f) for f in files],
                    'MB': [os.path.getsize(f) / 1e6 for f in files]}).sort_values('MB', ascending=False)
print(f'{len(files)} CSVs, {inv.MB.sum():,.0f} MB total'); inv.head(25).round(1)

## Step 1 — Define the plot set from `PLOT` (+ `PLOTGEOM` for the ecological section)

In [ ]:
def table_name(path):                       # 'WV_TREE.csv' -> 'TREE'
    return os.path.basename(path)[:-4].split('_', 1)[1]
path_of = {table_name(f): f for f in files}
assert 'PLOT' in path_of, 'WV_PLOT.csv not found in IN_DIR'

P = pd.read_csv(path_of['PLOT'], low_memory=False, dtype={'CN': str, 'PREV_PLT_CN': str})
n0 = len(P)
P = P[P.INVYR >= MIN_INVYR]
if SCOPE == 'SECTION':
    G = pd.read_csv(path_of['PLOTGEOM'], usecols=['CN', 'ECOSUBCD'], dtype={'CN': str})
    keep = G.loc[G.ECOSUBCD.astype(str).str.strip().str.startswith(SECTION), 'CN']
    P = P[P.CN.isin(keep)]
elif SCOPE == 'RADIUS':
    d = np.hypot((P.LAT - FERNOW_LATLON[0]) * 111.0, (P.LON - FERNOW_LATLON[1]) * 111.32 * np.cos(np.deg2rad(FERNOW_LATLON[0])))
    P = P[d <= RADIUS_KM]
PLOTS = set(P.CN)
print(f'PLOT: {n0:,} -> {len(P):,} plot visits kept (SCOPE={SCOPE}, INVYR >= {MIN_INVYR})')

## Step 2 — Filter every table (chunked), `TREE` first so tree-keyed tables can follow

In [ ]:
manifest, TREES = [], set()

def filter_table(name, path):
    t0 = time.time()
    if any(s in name for s in SKIP):
        manifest.append(dict(table=name, action='skipped (SKIP list)', rows_in=np.nan, rows_out=0,
                             MB_in=os.path.getsize(path) / 1e6, MB_out=0)); return
    header = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in COLS[name] if c in header] if name in COLS else header
    key = 'CN' if name in ('PLOT', 'PLOTGEOM', 'PLOTSNAP') else ('PLT_CN' if 'PLT_CN' in header else ('TRE_CN' if 'TRE_CN' in header else None))
    out = os.path.join(OUT_DIR, f'WV_{name}.csv.gz')
    mb_in = os.path.getsize(path) / 1e6
    if key is None:
        if mb_in <= COPY_WHOLE_MAX_MB:
            df = pd.read_csv(path, low_memory=False); df.to_csv(out, index=False, compression='gzip')
            manifest.append(dict(table=name, action='copied whole (no plot key)', rows_in=len(df), rows_out=len(df),
                                 MB_in=mb_in, MB_out=os.path.getsize(out) / 1e6))
        else:
            manifest.append(dict(table=name, action='skipped (no key, too large)', rows_in=np.nan, rows_out=0, MB_in=mb_in, MB_out=0))
        return
    keyset = PLOTS if key in ('CN', 'PLT_CN') else TREES
    n_in = n_out = 0; first = True
    dtypes = {c: str for c in usecols if c == 'CN' or c.endswith('_CN')}
    for ch in pd.read_csv(path, usecols=usecols, dtype=dtypes, chunksize=CHUNK, low_memory=False):
        n_in += len(ch)
        ch = ch[ch[key].isin(keyset)]
        if name == 'TREE':
            TREES.update(ch['CN'])
        n_out += len(ch)
        ch.to_csv(out, index=False, compression='gzip', mode='w' if first else 'a', header=first); first = False
    manifest.append(dict(table=name, action=f'filtered on {key}', rows_in=n_in, rows_out=n_out,
                         MB_in=mb_in, MB_out=os.path.getsize(out) / 1e6))
    print(f'{name:28s} {n_in:>11,} -> {n_out:>10,} rows   {mb_in:8.1f} -> {os.path.getsize(out) / 1e6:7.1f} MB   ({time.time() - t0:.0f} s)')

order = ['PLOT', 'TREE'] + [n for n in path_of if n not in ('PLOT', 'TREE')]
for name in order:
    if name in path_of:
        filter_table(name, path_of[name])
M = pd.DataFrame(manifest); M.to_csv(os.path.join(OUT_DIR, 'MANIFEST.csv'), index=False)
print(f"\nTotal: {M.MB_in.sum():,.0f} MB in -> {M.MB_out.sum():,.1f} MB out")
big = M[M.MB_out > 95]
if len(big):
    print('WARNING — over GitHub\'s 100 MB per-file limit (raise MIN_INVYR, narrow SCOPE, or keep local):', big.table.tolist())
M.round(1)

## Step 3 — Condition-level live-tree carbon (`fia_cond_live.csv.gz`)
FIADB `CARBON_AG`/`CARBON_BG` are lb C per tree; `TPA_UNADJ` expands a tree to trees per acre of *plot*. Dividing by the
condition's proportion of the matching plot component (`SUBPPROP_UNADJ` for trees ≥5.0 in, `MICRPROP_UNADJ` for
saplings 1.0–4.9 in) gives per acre **of condition**, then × 0.00112085 → Mg ha⁻¹. Large trees = DIA ≥ 19.7 in (50 cm),
matching the Fernow finding that most REF AGB sits in ≥50 cm stems. FIA biomass/carbon come from FIA's national
volume/biomass system, not Jenkins et al. (2003); expect a systematic offset from Fernow AGB-C.


In [ ]:
cond_path = os.path.join(OUT_DIR, 'WV_COND.csv.gz')
C = pd.read_csv(cond_path, dtype={'PLT_CN': str},
                usecols=lambda c: c in ['PLT_CN', 'CONDID', 'CONDPROP_UNADJ', 'SUBPPROP_UNADJ', 'MICRPROP_UNADJ', 'COND_STATUS_CD'])
parts = []
VALS = ['live_AGC', 'live_BGC', 'live_BA', 'live_TPA', 'large_AGC', 'snag_AGC']
for ch in pd.read_csv(os.path.join(OUT_DIR, 'WV_TREE.csv.gz'), dtype={'CN': str, 'PLT_CN': str}, chunksize=CHUNK):
    ch = ch[ch.TPA_UNADJ.notna() & ch.DIA.notna()].copy()
    live = (ch.STATUSCD == 1).values
    snag = ((ch.STATUSCD == 2) & (ch.STANDING_DEAD_CD == 1)).values if 'STANDING_DEAD_CD' in ch else (ch.STATUSCD == 2).values
    tpa = ch.TPA_UNADJ.values
    cag = ch.CARBON_AG.fillna(0).values * tpa; cbg = ch.CARBON_BG.fillna(0).values * tpa
    ch['live_AGC'] = np.where(live, cag, 0.0)
    ch['live_BGC'] = np.where(live, cbg, 0.0)
    ch['live_BA'] = np.where(live, 0.005454154 * ch.DIA.values ** 2 * tpa, 0.0)
    ch['live_TPA'] = np.where(live, tpa, 0.0)
    ch['large_AGC'] = np.where(live & (ch.DIA.values >= 19.7), cag, 0.0)
    ch['snag_AGC'] = np.where(snag, cag, 0.0)
    ch['grp'] = np.where(ch.DIA.values < 5.0, 'sap', 'tree')
    parts.append(ch.groupby(['PLT_CN', 'CONDID', 'grp'])[VALS].sum().reset_index())
T = pd.concat(parts).groupby(['PLT_CN', 'CONDID', 'grp']).sum().reset_index()
T = T.merge(C, on=['PLT_CN', 'CONDID'], how='left')
T['prop'] = np.where(T.grp == 'sap', T.MICRPROP_UNADJ, T.SUBPPROP_UNADJ)
T = T[T.prop > 0]
T[VALS] = T[VALS].div(T.prop, axis=0)
W = T.pivot_table(index=['PLT_CN', 'CONDID'], columns='grp',
                  values=['live_AGC', 'live_BGC', 'live_BA', 'live_TPA', 'large_AGC', 'snag_AGC'], fill_value=0)
W.columns = [f'{a}_{b}' for a, b in W.columns]; W = W.reset_index()
live = pd.DataFrame({'PLT_CN': W.PLT_CN, 'CONDID': W.CONDID,
    'live_AGC_MgHa': (W.get('live_AGC_tree', 0) + W.get('live_AGC_sap', 0)) * LBAC_TO_MGHA,
    'live_BGC_MgHa': (W.get('live_BGC_tree', 0) + W.get('live_BGC_sap', 0)) * LBAC_TO_MGHA,
    'large_AGC_MgHa': W.get('large_AGC_tree', 0) * LBAC_TO_MGHA,
    'snag_AGC_MgHa': (W.get('snag_AGC_tree', 0) + W.get('snag_AGC_sap', 0)) * LBAC_TO_MGHA,
    'live_BA_ft2ac': W.get('live_BA_tree', 0) + W.get('live_BA_sap', 0),
    'trees_per_ac_ge5in': W.get('live_TPA_tree', 0),
    'saplings_per_ac': W.get('live_TPA_sap', 0)})
live['large_tree_share'] = np.where(live.live_AGC_MgHa > 0, live.large_AGC_MgHa / live.live_AGC_MgHa, np.nan)
live.to_csv(os.path.join(OUT_DIR, 'fia_cond_live.csv.gz'), index=False, compression='gzip')
print(f'fia_cond_live: {len(live):,} plot-conditions'); live.describe().round(2)

## Step 4 — Condition-level seedlings by species group (`fia_cond_seedlings.csv.gz`)
FIA seedlings are counted on the 6.8-ft microplots (hardwoods ≥1 ft tall, <1.0 in DBH). Seedlings per acre of condition =
Σ(`TREECOUNT_CALC` × `TPA_UNADJ`) / `MICRPROP_UNADJ`. Species groups by FIA species code (SPCD).


In [ ]:
def spgroup(spcd):
    s = np.asarray(spcd)
    return np.select([(s >= 800) & (s <= 899), s == 316, s == 318, s == 315, s == 531, np.isin(s, [371, 372, 373, 375, 379]),
                      s == 762, s == 621, (s >= 400) & (s <= 499)],
                     ['oak', 'red_maple', 'sugar_maple', 'striped_maple', 'beech', 'birch', 'black_cherry', 'yellow_poplar', 'hickory'],
                     'other')
sp = os.path.join(OUT_DIR, 'WV_SEEDLING.csv.gz')
if os.path.exists(sp):
    S = pd.read_csv(sp, dtype={'PLT_CN': str}, low_memory=False)
    cnt = 'TREECOUNT_CALC' if 'TREECOUNT_CALC' in S else 'TREECOUNT'
    S['spa'] = S[cnt].fillna(0) * S.TPA_UNADJ.fillna(0)
    S['group'] = spgroup(S.SPCD)
    sd = S.pivot_table(index=['PLT_CN', 'CONDID'], columns='group', values='spa', aggfunc='sum', fill_value=0)
    sd.columns = [f'seedl_{c}' for c in sd.columns]; sd = sd.reset_index()
    sd = sd.merge(C[['PLT_CN', 'CONDID', 'MICRPROP_UNADJ']], on=['PLT_CN', 'CONDID'], how='left')
    sd = sd[sd.MICRPROP_UNADJ > 0]
    sc = [c for c in sd.columns if c.startswith('seedl_')]
    sd[sc] = sd[sc].div(sd.MICRPROP_UNADJ, axis=0)
    sd['seedl_total'] = sd[sc].sum(axis=1)
    sd.drop(columns='MICRPROP_UNADJ').to_csv(os.path.join(OUT_DIR, 'fia_cond_seedlings.csv.gz'), index=False, compression='gzip')
    print(f'fia_cond_seedlings: {len(sd):,} plot-conditions with seedlings recorded (conditions with none are absent = 0)')
    print(sd[sc + ['seedl_total']].median().round(0))
else:
    print('WV_SEEDLING not found — seedling summary skipped.')

## Done
Copy `OUT_DIR` into the repo (e.g. `Forest-Management-Carbon/FIA_WV_filtered/`) and push, then point `FIA_DIR` in
notebook 04 at that folder. Notebook 04 reads `.csv` or `.csv.gz` transparently. Check `MANIFEST.csv` for any file over
GitHub's 100 MB limit before pushing.
